# 1. Сбор данных мониторинга вузов

**Автор:** Шинкарев Сергей Дмитриевич, группа о.ИЗДт 23.2/Б3-22  
**Тема ВКР:** «Разработка системы интеллектуального анализа данных мониторинга вузов»

Блокнот показывает, **откуда берутся данные** и как они превращаются в набор для анализа.

Источник — открытые информационно-аналитические материалы мониторинга эффективности деятельности
образовательных организаций высшего образования (ГИВЦ Минобрнауки России): <https://monitoring.miccedu.ru/?m=vpo>.
Готовых файлов для скачивания сайт не предоставляет, поэтому данные извлекаются из HTML-страниц (веб-скрейпинг).
Случайные или искусственно сгенерированные данные в работе не используются.

In [1]:
# Подключаем пакет проекта: блокнот лежит в каталоге notebooks, код — уровнем выше
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

from vuzmon import config
from vuzmon.collect import parsers
from vuzmon.collect.http_client import CachedHttpClient
from vuzmon.collect.crawler import MonitoringCrawler, records_to_frames, load_frames, load_frames_from_url

## 1.1. Устройство источника

Материалы каждого года имеют три уровня страниц:

| Уровень | Адрес | Что извлекаем |
|---|---|---|
| оглавление года | `/?m=vpo&year=2025` | перечень субъектов РФ по федеральным округам |
| субъект РФ | `/iam/2025/_vpo/material.php?type=2&id=<код региона>` | перечень вузов и филиалов |
| организация | `/iam/2025/_vpo/inst.php?id=<код организации>` | паспорт, показатели E.1–E.8 за 5 лет, медианы, 56 дополнительных характеристик |

Скачивание выполняет «вежливый» HTTP-клиент: пауза между запросами, повтор при сбоях, дисковый кэш
(повторный запуск не обращается к сайту).

In [2]:
client = CachedHttpClient(config.CACHE_DIR, min_interval=0.3)
crawler = MonitoringCrawler(client, workers=4)

regions = crawler.fetch_regions(2025)
print("Субъектов РФ в мониторинге 2025 года:", len(regions))
pd.DataFrame([vars(r) for r in regions]).groupby("federal_district").size().rename("регионов").to_frame().T

Субъектов РФ в мониторинге 2025 года: 89


federal_district,ДФО,Новые регионы,ПФО,СЗФО,СКФО,СФО,УФО,ЦФО,ЮФО
регионов,11,4,14,11,7,10,6,18,8


In [3]:
# Организации одного региона: г. Москва
moscow = next(r for r in regions if "Москва" in r.name)
refs = crawler.fetch_region_institutions(2025, moscow)
print(f"{moscow.name}: организаций — {len(refs)}, из них филиалов — {sum(r.is_branch for r in refs)}")
pd.DataFrame([vars(r) for r in refs]).head(5)

г.Москва: организаций — 151, из них филиалов — 3


,inst_id,name,city,is_branch,longitude,latitude
0,111037,Автономная некоммерческая образовательная орга...,г. Москва,False,37.648636,55.760175
1,113493,Автономная некоммерческая образовательная орга...,г. Москва,False,37.359576,55.698538
2,113596,"Автономная некоммерческая организация ""Образов...",г. Москва,False,37.600945,55.828865
3,112814,Автономная некоммерческая организация высшего ...,г. Москва,False,37.618686,55.796238
4,110188,Автономная некоммерческая организация высшего ...,г. Москва,False,37.646399,55.801055


## 1.2. Разбор страницы организации

В качестве примера — страница Московского университета имени С.Ю. Витте (код организации в мониторинге — 1943).

In [4]:
ref = next(r for r in refs if r.inst_id == 1943)
record = crawler.fetch_institution(2025, ref)
print(record.info["name"])
print("Ведомственная принадлежность:", record.info["founder_group"])
pd.DataFrame([{"код": m.code, "показатель": m.title, **m.values, "медиана РФ": m.median_rf,
               "медиана региона": m.median_region} for m in record.main])

Частное образовательное учреждение высшего образования "Московский университет им. С.Ю. Витте"
Ведомственная принадлежность: Частные организации


,код,показатель,2021,2022,2023,2024,2025,медиана РФ,медиана региона
0,E.1,Образовательная деятельность,59.91,47.30,55.37,57.32,54.43,62.80,69.08
1,E.2,Научно-исследовательская деятельность,161.04,204.71,147.65,139.13,181.82,132.10,333.80
2,E.3,Международная деятельность,4.10,3.64,3.41,2.61,2.72,4.91,6.10
3,E.4,Финансово-экономическая деятельность,10396.66,11888.80,12500.18,13846.99,13635.84,4241.60,7012.90
4,E.8,Дополнительный показатель,0.65,0.53,0.47,0.50,0.49,3.13,3.91


In [5]:
extra = pd.DataFrame([vars(e) for e in record.extra])
print("Дополнительных характеристик:", len(extra))
extra.groupby("section").size().rename("показателей").to_frame().T

Дополнительных характеристик: 59


section,Инфраструктура,Кадровый потенциал,Международная деятельность,Научная деятельность,Образовательная деятельность,Финансово-экономическая деятельность
показателей,9,9,9,11,12,9


In [6]:
extra.head(8)

,code,section,name,unit,value
0,1,Образовательная деятельность,"Общая численность студентов, обучающихся по пр...",чел.,15716.00
1,1.1,Образовательная деятельность,по очной форме обучения,чел.,2233.00
2,1.2,Образовательная деятельность,по очно-заочной (вечерней) форме обучения,чел.,6846.00
3,1.3,Образовательная деятельность,по заочной форме обучения,чел.,6637.00
4,2,Образовательная деятельность,"Средний балл ЕГЭ студентов, принятых на обучен...",балл,55.92
5,3,Образовательная деятельность,"Доля обучающихся по программам бакалавриата, с...",%,14.21
6,4,Образовательная деятельность,Доля обучающихся по программам бакалавриата в ...,%,83.78
7,5,Образовательная деятельность,Доля обучающихся по программам магистратуры в ...,%,10.36


## 1.3. Пробный сбор и формирование таблиц

Полный сбор за 2021–2025 годы — это около 6,5 тысячи страниц и примерно 30 минут. Он запускается командой

```
python -m vuzmon.collect --years 2021 2022 2023 2024 2025
```

или кнопкой «Запустить сбор» в разделе «Данные» веб-приложения. Здесь для демонстрации собираются 12 организаций.

In [7]:
result = crawler.crawl_year(2025, limit=12)
demo = records_to_frames(result.records)
print({name: frame.shape for name, frame in demo.items()}, "| ошибок:", len(result.errors))
demo["institutions"][["year", "inst_id", "name", "is_branch", "region", "federal_district", "founder_group"]].head(6)

{'institutions': (12, 28), 'main': (60, 11), 'medians': (60, 6), 'extra': (12, 61), 'dictionary': (64, 5)} | ошибок: 0


,year,inst_id,name,is_branch,region,federal_district,founder_group
0,2025,1,Федеральное государственное бюджетное образова...,0,Республика Адыгея,ЮФО,Министерство науки и высшего образования Росси...
1,2025,2,Федеральное государственное бюджетное образова...,0,Алтайский край,СФО,Министерство просвещения Российской Федерации
2,2025,3,Федеральное государственное бюджетное образова...,0,Алтайский край,СФО,Министерство науки и высшего образования Росси...
3,2025,4,Федеральное государственное бюджетное образова...,0,Алтайский край,СФО,Министерство науки и высшего образования Росси...
4,2025,5,Федеральное государственное бюджетное образова...,0,Амурская область,ДФО,Министерство науки и высшего образования Росси...
5,2025,6,Федеральное государственное бюджетное образова...,0,Хабаровский край,ДФО,Министерство просвещения Российской Федерации


## 1.4. Итоговый набор данных

Собранный набор хранится в репозитории (`data/monitoring`) в виде пяти CSV-файлов. Его можно прочитать
с диска либо **напрямую по ссылке** из публичного репозитория:
<https://github.com/sinkarev39/vuz-monitoring-analysis/tree/main/data/monitoring>. Тем же способом набор загружается
в разделе «Данные» веб-приложения.

In [8]:
# Набор данных опубликован в репозитории проекта; его адрес хранится в config.DATASET_URL.
# LOAD_FROM_URL = True — скачать набор напрямую по ссылке, False — прочитать файлы из каталога data/monitoring
DATASET_URL = config.DATASET_URL
LOAD_FROM_URL = False

frames = load_frames_from_url(DATASET_URL) if LOAD_FROM_URL else load_frames(config.MONITORING_DIR)
pd.DataFrame({"файл": ["institutions.csv", "main_indicators.csv", "medians.csv", "extra_indicators.csv", "indicator_dictionary.csv"],
              "содержание": ["паспорта организаций по годам", "основные показатели E.1–E.8 (динамика)", "медианы по РФ, региону, ведомству",
                             "56 дополнительных характеристик", "справочник показателей"],
              "строк": [len(frames[k]) for k in ("institutions", "main", "medians", "extra", "dictionary")],
              "столбцов": [frames[k].shape[1] for k in ("institutions", "main", "medians", "extra", "dictionary")]})

,файл,содержание,строк,столбцов
0,institutions.csv,паспорта организаций по годам,6127,28
1,main_indicators.csv,основные показатели E.1–E.8 (динамика),30400,11
2,medians.csv,"медианы по РФ, региону, ведомству",35227,6
3,extra_indicators.csv,56 дополнительных характеристик,6127,61
4,indicator_dictionary.csv,справочник показателей,66,5


In [9]:
inst = frames["institutions"]
summary = inst.groupby("year").agg(организаций=("inst_id", "count"), филиалов=("is_branch", "sum"), регионов=("region_id", "nunique"))
summary["головных"] = summary["организаций"] - summary["филиалов"]
print("Уникальных организаций за все годы:", inst["inst_id"].nunique())
summary

Уникальных организаций за все годы: 1352


,организаций,филиалов,регионов,головных
year,,,,
2021,1222,530,84,692
2022,1208,510,84,698
2023,1206,510,84,696
2024,1234,525,88,709
2025,1257,521,88,736


## 1.5. Выводы

* Данные получены из первоисточника самостоятельно, методом веб-скрейпинга; набор воспроизводим — сборщик входит в проект.
* Источник слабо структурирован: значения показателей, медианы и справочные сведения разбросаны по нескольким HTML-таблицам,
  числа записаны в российском формате, часть показателей есть не у всех организаций. Разборщик приводит всё к единому виду.
* В ходе работы обнаружена и исправлена особенность источника: в материалах 2021–2023 годов не подписаны федеральные округа,
  округ восстанавливается по коду региона из материалов других лет.
* Итог: 6 127 записей «организация — год» за 2021–2025 годы, динамика основных показателей с 2017 года.